# 03 · 决策树、随机森林与 XGBoost Decision Trees, Random Forest & XGBoost

> **能力标签**：SH7（经典机器学习 / Classical ML）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解**决策树**（decision tree）的分裂准则（Gini 不纯度、信息熵）与过拟合风险。
2. 掌握**随机森林**（random forest）通过 Bagging + 特征随机化降低方差的机制。
3. 理解**梯度提升**（gradient boosting）逐步拟合残差的思想，以及 **XGBoost** 的 `hist` 加速算法。
4. 使用 `XGBClassifier(tree_method="hist")` 并以 ROC-AUC 评估。

> 对应能力：**SH7**

## 概念讲解 Concepts

### 决策树 Decision Tree

决策树通过**递归二分**（recursive binary splitting）构建：每步选择使目标函数（不纯度）下降最多的特征与阈值。

| 准则 | 公式 | 用途 |
|------|------|------|
| **Gini 不纯度** | $1 - \sum_k p_k^2$ | 分类（sklearn 默认） |
| **信息熵** | $-\sum_k p_k \log_2 p_k$ | 分类 |
| **MSE** | $\text{Var}(y)$ | 回归 |

**问题**：单棵深树极易过拟合。解决方案：剪枝（`max_depth`）或集成。

---

### 随机森林 Random Forest（Bagging）

- 对训练集做 $B$ 次**有放回采样**（bootstrap）。
- 每棵树分裂时只考虑 $\sqrt{p}$ 个随机特征（feature subsampling）。
- 最终预测为 $B$ 棵树的**多数投票**（分类）或**均值**（回归）。

核心：树间**相关性低** → 集成误差 $\approx$ 单树偏差 + 方差/B。

---

### 梯度提升 Gradient Boosting & XGBoost

梯度提升**串行**构建树，每棵树拟合前一轮的**负梯度**（伪残差）：

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

**XGBoost** 在此基础上：
- 使用**二阶泰勒展开**近似损失。
- `tree_method="hist"`：直方图近似，大数据集下速度极快。
- 内置正则化（$L_1/L_2$ 叶权重惩罚）。

## 示例 Worked Example — `fit_xgb_auc`

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# ── fit_xgb_auc（镜像 w5-xgb 题包）─────────────────────────────────────────
def fit_xgb_auc(Xtr, ytr, Xte, yte) -> float:
    """训练 XGBoost 分类器（hist），返回测试集 ROC-AUC。"""
    clf = XGBClassifier(n_estimators=50, max_depth=3, tree_method="hist",
                        eval_metric="logloss", random_state=0)
    clf.fit(Xtr, ytr)
    return float(roc_auc_score(yte, clf.predict_proba(Xte)[:, 1]))

# ── 数据集 ────────────────────────────────────────────────────────────────
X, y = make_classification(n_samples=600, n_features=20, n_informative=8,
                            n_redundant=4, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

# ── 三种模型比较 ───────────────────────────────────────────────────────────
results = {}

# 决策树
dt = DecisionTreeClassifier(max_depth=4, random_state=0)
dt.fit(Xtr, ytr)
results["DecisionTree (depth=4)"] = roc_auc_score(yte, dt.predict_proba(Xte)[:, 1])

# 随机森林
rf = RandomForestClassifier(n_estimators=100, random_state=0)
rf.fit(Xtr, ytr)
results["RandomForest (100 trees)"] = roc_auc_score(yte, rf.predict_proba(Xte)[:, 1])

# XGBoost (hist)
results["XGBoost hist (50 trees)"] = fit_xgb_auc(Xtr, ytr, Xte, yte)

print(f"{'模型':<30}  {'ROC-AUC':>9}")
print("-" * 42)
for name, auc in results.items():
    print(f"{name:<30}  {auc:.4f}")

print("\n✓ XGBoost 通常在结构化数据上优于单棵决策树。")

## 动手 Exercise

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# 练习：调节 n_estimators 观察 RF 与 XGBoost 的 AUC 变化
X, y = make_classification(n_samples=800, n_features=25, n_informative=10,
                            random_state=99)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)

print(f"{'模型':<25}  {'n_est':>6}  {'AUC':>8}")
print("-" * 45)
for n_est in [10, 50, 200]:
    rf  = RandomForestClassifier(n_estimators=n_est, random_state=0)
    rf.fit(Xtr, ytr)
    auc_rf = roc_auc_score(yte, rf.predict_proba(Xte)[:, 1])
    print(f"{'RandomForest':<25}  {n_est:>6}  {auc_rf:.4f}")

    xgb = XGBClassifier(n_estimators=n_est, tree_method="hist",
                        eval_metric="logloss", random_state=0)
    xgb.fit(Xtr, ytr)
    auc_xgb = roc_auc_score(yte, xgb.predict_proba(Xte)[:, 1])
    print(f"{'XGBoost(hist)':<25}  {n_est:>6}  {auc_xgb:.4f}")

print("\n观察：增加 n_estimators 通常提升 AUC，但边际收益递减。")

## 延伸阅读 Further Reading

1. **XGBoost 原论文**："XGBoost: A Scalable Tree Boosting System" — Chen & Guestrin (KDD 2016)
2. **sklearn 集成方法**：<https://scikit-learn.org/stable/modules/ensemble.html>
3. **StatQuest — Random Forests**：<https://www.youtube.com/watch?v=J4Wdy0Wc_xQ>
4. **《The Elements of Statistical Learning》** — 第 10 章（Boosting 与加性树）

## 关联练习 Related Assignments

```bash
python check.py w5-xgb
```

> 实现 `fit_xgb_auc(Xtr, ytr, Xte, yte)` — 训练 XGBoost (`tree_method="hist"`)，返回测试集 ROC-AUC。

> 能力标签：**SH7** · threshold ≥ 0.7